# Exploración: SUNARP — Servicio de Publicidad Registral en Línea (SPRL)

- **Entidad oficial:** Superintendencia Nacional de los Registros Públicos (SUNARP).
- **Descripción y propósito:** Plataforma transaccional autenticada del Estado peruano que permite acceder a los servicios de publicidad registral, búsqueda de índices y visualización de partidas registrales (Personas Jurídicas, Propiedad Inmueble, Mandatos y Poderes, etc.).
- **Enlaces del flujo SSO:**
  - Portada de ingreso: `https://sprl.sunarp.gob.pe/sprl/ingreso`
  - Pasarela de autorización (SSO): `https://im01-autorizacion-sprl-production.apps.paas.sunarp.gob.pe/v1/sunarp-services/im/autorizacion/login`
  - Módulo objetivo (Visualización de partidas): `https://sprl.sunarp.gob.pe/sprl/main/partidas-base-grafica-registral`
- **Mecanismo de acceso:** Autenticación mediante cuenta de usuario registrada (`SUNARP_USER` y `SUNARP_PASS`) protegida con desafío interactivo de **Cloudflare Turnstile**.
- **Resultado del módulo de autenticación:** Obtención y persistencia en disco (`sunarp_sprl_auth.json` + perfil nativo `./chrome_sprl_profile`) de las cookies de sesión (`cf_clearance`, cookies de aplicación) y del estado de `localStorage` (tokens JWT de los orígenes de SUNARP) para reutilizarlos en consultas posteriores sin repetir el login.

## Estrategia Técnica de Autenticación y Evasión Anti-Bot

- **Navegador con contexto persistente (`launch_persistent_context`):**
  1. Se levanta Google Chrome nativo (`channel="chrome"`) sobre un directorio de perfil dedicado (`./chrome_sprl_profile`), sin sobrescribir el `User-Agent` real del binario para mantener consistencia total con los *Client Hints* (`sec-ch-ua`) evaluados por Cloudflare.
  2. Se desactiva la bandera `--no-sandbox` por defecto de Playwright (`ignore_default_args=["--no-sandbox"]`) y se oculta `navigator.webdriver`.
- **Orquestación del flujo Single Sign-On (SSO):**
  1. Se navega inicialmente a la ruta privada objetivo (`/sprl/main/partidas-base-grafica-registral`). Si no existe sesión activa, el portal redirige a `/sprl/ingreso`.
  2. Se cierra automáticamente el modal publicitario emergente (*"Sunarp Simplifica"*) mediante evento `Escape`, clic en el botón de cierre o limpieza del contenedor overlay en el DOM, y se acciona el botón `INGRESAR` de la portada para generar el contexto de redirección SSO válido.
- **Resolución automatizada de Cloudflare Turnstile:**
  1. Se completan las credenciales con tipeo secuencial (`press_sequentially`) simulando cadencia humana.
  2. Se monitorea el `input[name="cf-turnstile-response"]` y, en caso de requerir interacción sobre la casilla *"Verifique que es un ser humano"*, se calcula dinámicamente la caja delimitadora (`bounding_box`) del widget para ejecutar un clic físico con trayectoria de ratón (`page.mouse.move` + `page.mouse.click`).
- **Persistencia de estado (`storage_state`):**
  - Una vez completada la redirección automática hacia `/sprl/main/partidas-base-grafica-registral`, se exportan las cookies y los orígenes (`localStorage`) a `sunarp_sprl_auth.json`.

In [8]:
import asyncio
import json
import os
import re
from pathlib import Path
from playwright.async_api import async_playwright
import ddddocr

# 1. Configuración General
URL_OBJETIVO_SPRL = "https://sprl.sunarp.gob.pe/sprl/main/partidas-base-grafica-registral"

SUNARP_USER = os.getenv("SUNARP_USER", "MYUSER123")
SUNARP_PASS = os.getenv("SUNARP_PASS", "BU&VQESCG9")

PARAM_OFICINA = "CUSCO"
PARAM_AREA = "Personas Jurídicas"
PARAM_PARTIDA = "11156404"

ARCHIVO_SESION = Path("sunarp_sprl_auth.json")
PERFIL_DIR = Path("./chrome_sprl_profile")

# Inicializamos el motor neuronal una sola vez en memoria (super rápido)
ocr = ddddocr.DdddOcr(show_ad=False)

# 2. Funciones Auxiliares
def resolver_captcha_imagen(imagen_bytes: bytes) -> str:
    """Resuelve el CAPTCHA usando la red neuronal ddddocr."""
    texto = ocr.classification(imagen_bytes)
    return re.sub(r'[^A-Z0-9]', '', texto.upper())

async def cerrar_modal_publicitario(page):
    await page.wait_for_timeout(1000)
    await page.keyboard.press("Escape")
    await page.wait_for_timeout(400)
    selectores_x = ["nz-modal-close", "button[aria-label*='Close' i]", "span:has-text('×')"]
    for sel in selectores_x:
        loc = page.locator(sel).first
        if await loc.count() > 0 and await loc.is_visible():
            try:
                await loc.click(timeout=1000)
                await page.wait_for_timeout(500)
                return
            except Exception:
                pass
    await page.mouse.click(15, 15)

async def clicar_checkbox_turnstile(page) -> bool:
    selectores = ["iframe[src*='cloudflare']", "iframe[src*='turnstile']", "div.cf-turnstile"]
    for sel in selectores:
        loc = page.locator(sel).first
        if await loc.count() > 0 and await loc.is_visible():
            box = await loc.bounding_box()
            if box:
                x = box["x"] + 32
                y = box["y"] + (box["height"] / 2)
                await page.mouse.move(x, y, steps=10)
                await page.wait_for_timeout(150)
                await page.mouse.click(x, y)
                return True
    return False

# 3. Flujo Principal Unificado
async def consulta_sunarp_unificada() -> dict:
    async with async_playwright() as p:
        modo_oculto = os.getenv("DOCKER_ENV", "false").lower() == "true"
        
        context = await p.chromium.launch_persistent_context(
            user_data_dir=str(PERFIL_DIR),
            headless=modo_oculto,
            channel="chrome",
            viewport={"width": 1366, "height": 768},
            locale="es-PE",
            ignore_default_args=["--no-sandbox"],
            args=["--disable-blink-features=AutomationControlled"]
        )
        await context.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        page = context.pages[0] if context.pages else await context.new_page()

        try:
            # === FASE A: LOGIN ===
            print(f"[1/7] Evaluando estado de sesión en: {URL_OBJETIVO_SPRL}...")
            await page.goto(URL_OBJETIVO_SPRL, wait_until="domcontentloaded", timeout=30000)
            
            for _ in range(20):
                texto_body = await page.inner_text("body")
                if "sprl/ingreso" in page.url or "autorizacion" in page.url or "Hola," in texto_body or await page.locator("input[type='password']").count() > 0:
                    break
                await page.wait_for_timeout(500)

            if "Hola," not in await page.inner_text("body") and "Búsqueda Directa" not in await page.inner_text("body"):
                print("[2/7] Sesión no detectada. Iniciando flujo de login SSO...")
                if "sprl/ingreso" in page.url:
                    await cerrar_modal_publicitario(page)
                    btn_portada = page.locator("button:has-text('INGRESAR'), a:has-text('INGRESAR')").first
                    if await btn_portada.count() > 0:
                        await btn_portada.click()
                        await page.wait_for_load_state("domcontentloaded")
                        await page.wait_for_timeout(1500)

                input_user = page.locator("input[placeholder*='usuario' i], input[type='text']").first
                input_pass = page.locator("input[placeholder*='contraseña' i], input[type='password']").first
                await input_user.wait_for(state="visible", timeout=15000)
                await input_user.fill(SUNARP_USER)
                await input_pass.fill(SUNARP_PASS)

                print("   -> Resolviendo Turnstile de Login...")
                await page.wait_for_timeout(1000)
                for intento in range(15):
                    if await page.evaluate("() => { const el = document.querySelector('input[name=\"cf-turnstile-response\"]'); return el ? el.value : ''; }"):
                        break
                    if intento in (2, 6, 10):
                        await clicar_checkbox_turnstile(page)
                    await page.wait_for_timeout(1000)

                await page.locator("button:has-text('INGRESAR')").first.click()
                await page.locator("text=/Hola,/i").first.wait_for(state="visible", timeout=25000)

            if "partidas-base-grafica-registral" not in page.url:
                await page.goto(URL_OBJETIVO_SPRL, wait_until="domcontentloaded", timeout=30000)
            
            await page.locator("text=Búsqueda Directa por Número de Partida").first.wait_for(state="visible", timeout=20000)
            await page.wait_for_timeout(1000)
            await context.storage_state(path=str(ARCHIVO_SESION))

            # === FASE B: EXTRACCIÓN Y BÚSQUEDA ===
            print("[3/7] Resolviendo Cloudflare Turnstile interno...")
            for intento in range(15):
                if await page.evaluate("() => { const el = document.querySelector('input[name=\"cf-turnstile-response\"]'); return el ? el.value : ''; }"):
                    break
                if intento in (2, 5, 8):
                    await clicar_checkbox_turnstile(page)
                await page.wait_for_timeout(1000)

            print(f"[4/7] Seleccionando Oficina: {PARAM_OFICINA} y Área: {PARAM_AREA}...")
            cajas_select = page.locator("nz-select")
            await cajas_select.first.wait_for(state="visible", timeout=15000)

            # 1. Oficina
            print("   -> Desplegando Oficina Registral...")
            await cajas_select.nth(0).click()
            await page.wait_for_timeout(1000)
            
            await page.keyboard.type(PARAM_OFICINA, delay=50)
            await page.wait_for_timeout(800)
            
            opc_oficina = page.locator(f"nz-option-item[title*='{PARAM_OFICINA}' i]").first
            if await opc_oficina.count() > 0:
                await opc_oficina.evaluate("el => el.click()")
            else:
                await page.keyboard.press("Enter")
            await page.wait_for_timeout(1000)

            # 2. Área
            print("   -> Desplegando Área Registral...")
            await cajas_select.nth(1).click()
            await page.wait_for_timeout(1000)
            
            param_area_safe = "Personas Jur" 
            await page.keyboard.type(param_area_safe, delay=50)
            await page.wait_for_timeout(800)
            
            opc_area = page.locator(f"nz-option-item[title*='{param_area_safe}' i]").first
            if await opc_area.count() > 0:
                await opc_area.evaluate("el => el.click()")
            else:
                await page.keyboard.press("Enter")
                
            await page.wait_for_timeout(2500) 

            # 3. Radio Partida (INMUNE AL SCROLL)
            print("   -> Seleccionando opción 'Partida'...")
            radio_partida = page.locator("label:has-text('Partida')").first
            await radio_partida.wait_for(state="attached", timeout=5000)
            
            # Forzamos scroll visual para depuración y obligamos el clic mediante JavaScript
            await radio_partida.scroll_into_view_if_needed()
            await page.wait_for_timeout(800)
            await radio_partida.evaluate("el => el.click()")
                
            await page.wait_for_timeout(1500)

            # 4. Número
            print(f"   -> Ingresando número de partida: {PARAM_PARTIDA}...")
            input_numero = page.locator("#numero").first
            await input_numero.wait_for(state="visible", timeout=5000)
            await input_numero.click()
            await page.keyboard.press("Control+A") 
            await page.keyboard.press("Backspace")
            await input_numero.fill("")
            await input_numero.press_sequentially(PARAM_PARTIDA, delay=50)
            await input_numero.press("Tab")  
            await page.wait_for_timeout(500)

            print("[5/7] Resolviendo CAPTCHA con Red Neuronal ddddocr...")
            contenedor_captcha = page.locator("app-captcha-sunarp").first
            img_captcha_locator = contenedor_captcha.locator("img.img-captcha").first
            input_captcha = contenedor_captcha.locator("input").first
            btn_recargar = contenedor_captcha.locator("a[color='warn']").first
            btn_buscar = page.locator("button:has-text('Buscar')").first

            max_reintentos = 15
            tabla_encontrada = False

            for intento in range(1, max_reintentos + 1):
                await page.wait_for_timeout(1500)
                await img_captcha_locator.wait_for(state="visible", timeout=10000)
                
                img_bytes = await img_captcha_locator.screenshot()
                texto_captcha = resolver_captcha_imagen(img_bytes)
                print(f"   -> [Intento {intento}/{max_reintentos}] ddddocr detectó: '{texto_captcha}'")

                if len(texto_captcha) != 6:
                    print(f"   -> Longitud incorrecta ({len(texto_captcha)}/6). Recargando...")
                    if await btn_recargar.is_visible(): 
                        await btn_recargar.click(force=True)
                    continue

                await input_captcha.click()
                await page.keyboard.press("Control+A") 
                await page.keyboard.press("Backspace")
                await input_captcha.fill("")
                
                await input_captcha.press_sequentially(texto_captcha, delay=50)
                await input_captcha.press("Enter")
                await page.wait_for_timeout(500)
                
                await btn_buscar.click(force=True)

                try:
                    respuesta = await page.wait_for_selector(".ant-table, table, nz-modal-confirm-container, .ant-modal-wrap, .ant-message", timeout=6000)
                    clase_res = await respuesta.get_attribute("class") or ""
                    tag_name = await respuesta.evaluate("el => el.tagName.toLowerCase()")
                    
                    if "modal" in clase_res or "modal" in tag_name:
                        print("   -> Error SUNARP (Modal Detectado). Cerrando modal...")
                        btn_aceptar = page.locator("button:has-text('Aceptar'), button:has-text('OK')").first
                        if await btn_aceptar.is_visible():
                            await btn_aceptar.click()
                        await page.wait_for_timeout(2000)
                            
                    elif "message" in clase_res or "notification" in clase_res:
                        print("   -> Error SUNARP (Alerta pequeña). Recargando manualmente...")
                        if await btn_recargar.is_visible(): 
                            await btn_recargar.click(force=True)
                        await page.wait_for_timeout(1000)
                            
                    else:
                        print("   -> ¡Captcha correcto! Tabla principal cargada exitosamente.")
                        tabla_encontrada = True
                        break
                        
                except Exception:
                    if await btn_recargar.is_visible(): 
                        await btn_recargar.click(force=True)

            if not tabla_encontrada:
                return {"error": "Se superó el límite de reintentos del CAPTCHA OCR sin éxito."}

            print("[6/7] Extrayendo datos de la tabla principal...")
            await page.wait_for_timeout(1000)
            filas_locator = page.locator("table tbody tr")
            resultados_tabla = []

            for i in range(await filas_locator.count()):
                celdas = filas_locator.nth(i).locator("td")
                if await celdas.count() >= 8:
                    resultados_tabla.append({
                        "zona_registral": await celdas.nth(0).inner_text(),
                        "oficina_registral": await celdas.nth(1).inner_text(),
                        "partida": await celdas.nth(2).inner_text(),
                        "numero_ficha": await celdas.nth(3).inner_text(),
                        "tomo": await celdas.nth(4).inner_text(),
                        "folio": await celdas.nth(5).inner_text(),
                        "area_registral": await celdas.nth(6).inner_text(),
                        "registro_de": await celdas.nth(7).inner_text(),
                    })

            # === FASE C: EXTRACCIÓN DE ASIENTOS (CORREGIDA) ===
            titulos_asientos = []
            try:
                print("[7/7] Abriendo 'Lista de Asientos' y extrayendo Títulos...")
                
                btn_visualizar = page.locator("app-button#tabla button").first
                await btn_visualizar.wait_for(state="attached", timeout=10000)
                await btn_visualizar.evaluate("el => el.click()")
                
                await page.wait_for_selector("text='Lista de Asientos'", state="visible", timeout=15000)
                await page.wait_for_timeout(2000)
                
                elementos_titulo = page.locator("div.bg-gray").filter(has_text=re.compile(r"Titulo", re.IGNORECASE))
                
                cantidad_titulos = await elementos_titulo.count()
                for i in range(cantidad_titulos):
                    texto_bruto = await elementos_titulo.nth(i).inner_text()
                    
                    match = re.search(r"Titulo\s*:\s*(\d{4})\s*-\s*(\d+)", texto_bruto, re.IGNORECASE)
                    if match:
                        titulos_asientos.append({
                            "anio": match.group(1),
                            "numero": match.group(2),
                            "texto_original": match.group(0)
                        })
                        
                print(f"   -> Se extrajeron {len(titulos_asientos)} asientos correctamente.")
                
            except Exception as e:
                print(f"   -> Advertencia: No se pudo extraer la lista de asientos. Detalle: {e}")

            return {
                "audit": {
                    "source": "SUNARP_SPRL",
                    "oficina": PARAM_OFICINA,
                    "partida": PARAM_PARTIDA,
                    "status": "SUCCESS" if resultados_tabla else "NO_DATA"
                },
                "data": resultados_tabla,
                "asientos": titulos_asientos
            }

        except Exception as exc:
            return {"error": str(exc)}
        finally:
            await context.close()

# Ejecución nativa para Jupyter Notebook
resultado_extraccion = await consulta_sunarp_unificada()
print("\n================== RESULTADO FINAL SUNARP ==================")
print(json.dumps(resultado_extraccion, indent=2, ensure_ascii=False))

[1/7] Evaluando estado de sesión en: https://sprl.sunarp.gob.pe/sprl/main/partidas-base-grafica-registral...
[2/7] Sesión no detectada. Iniciando flujo de login SSO...
   -> Resolviendo Turnstile de Login...

================== RESULTADO FINAL SUNARP ==================
{
  "error": "Locator.wait_for: Timeout 25000ms exceeded.\nCall log:\n  - waiting for locator(\"text=/Hola,/i\").first to be visible\n"
}
